In [1]:
import os
import numpy as np
from scipy.ndimage import gaussian_filter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, random_split, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import math
import random
from tqdm.auto import tqdm

In [2]:
!nvidia-smi

Fri Apr  3 16:23:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:E3:00.0 Off |                    0 |
| N/A   29C    P0             92W /  500W |   66883MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
DATA_DIR  = "/mimer/NOBACKUP/groups/caim1/dafne/datasets/smooth_synthetic"
SAVE_DIR  = '/mimer/NOBACKUP/groups/caim1/dafne/checkpoints'
VIS_DIR   = '/mimer/NOBACKUP/groups/caim1/dafne/visual'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(VIS_DIR,  exist_ok=True)

In [4]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class SyntheticDVFDataset(Dataset):
    def __init__(self, root_dir, split="train"):
        self.field_dir = os.path.join(root_dir, split, "b")
        self.slice_dir = os.path.join(root_dir, split, "a")
        self.ids = sorted([
            f.split("_")[1].split(".")[0]
            for f in os.listdir(self.field_dir)
            if f.startswith("field_")
        ])
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, idx):
        i = self.ids[idx]
        dvf    = np.load(os.path.join(self.field_dir, f"field_{i}.npy"))
        slices = np.load(os.path.join(self.slice_dir,  f"slice_{i}.npy"),
                         allow_pickle=True).item()
        cond_coronal  = slices["coronal"]
        cond_sagittal = slices["sagittal"]
        mid_d         = slices["indices"]["mid_d"]
        dvf           = torch.from_numpy(dvf).float()
        cond_coronal  = torch.from_numpy(cond_coronal).float()
        cond_sagittal = torch.from_numpy(cond_sagittal).float()
        D         = dvf.shape[-1]
        slice_pos = mid_d / (D - 1)
        return cond_coronal, cond_sagittal, dvf, slice_pos

train_dataset = SyntheticDVFDataset(DATA_DIR, split="train")
test_dataset  = SyntheticDVFDataset(DATA_DIR, split="test")
print(f"Train samples: {len(train_dataset)}")
print(f"Test  samples: {len(test_dataset)}")
cor, sag, dvf, sp = train_dataset[0]
print(f"cond_coronal:  {tuple(cor.shape)}")   # expect (3, 128, 128) — W=256, D=128 → mid slices
print(f"cond_sagittal: {tuple(sag.shape)}")   # expect (3, 256, 128) — H=256, D=128
print(f"dvf:           {tuple(dvf.shape)}")   # expect (3, 256, 256, 128)
print(f"slice_pos:     {sp:.4f}")             # expect 0.5

Train samples: 500
Test  samples: 100
cond_coronal:  (3, 256, 128)
cond_sagittal: (3, 256, 128)
dvf:           (3, 256, 256, 128)
slice_pos:     0.5039


In [5]:
# ── Splits & loaders ──────────────────────────────────────────────────────────
train_dataset = SyntheticDVFDataset(DATA_DIR, split="train")
val_size      = int(0.15 * len(train_dataset))
train_size    = len(train_dataset) - val_size
train_dataset, val_dataset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
test_dataset = SyntheticDVFDataset(DATA_DIR, split="test")

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,    # ← was 2
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=2, shuffle=False,   # ← was 2
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")

Train=425, Val=75, Test=100


In [6]:
# ── Model ─────────────────────────────────────────────────────────────────────
def linear_beta_schedule(timesteps, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, timesteps)


class DiffusionSchedule:
    def __init__(self, timesteps=1000, device="cpu"):
        self.timesteps = timesteps
        self.device    = device
        self.betas     = torch.linspace(1e-4, 0.02, timesteps).to(device)
        self.alphas    = 1.0 - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)

        self.sqrt_alphas_bar           = torch.sqrt(self.alpha_bar)
        self.sqrt_one_minus_alphas_bar = torch.sqrt(1. - self.alpha_bar)

    def q_sample(self, x0, t, noise):
        sqrt_ab           = self.sqrt_alphas_bar[t][:, None, None, None, None]
        sqrt_one_minus_ab = self.sqrt_one_minus_alphas_bar[t][:, None, None, None, None]
        return sqrt_ab * x0 + sqrt_one_minus_ab * noise


class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb = torch.exp(
            torch.arange(half_dim, device=t.device) * -(math.log(10000) / (half_dim - 1))
        )
        emb = t[:, None] * emb[None, :] * 2 * math.pi   # ← ADD THIS
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)


class SliceToVolume(nn.Module):
    def __init__(self, out_channels=16):
        super().__init__()
        self.coronal_encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, out_channels, 3, padding=1)
        )
        self.sagittal_encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, out_channels, 3, padding=1)
        )
        # single 3D conv to learn interactions between coronal and sagittal
        self.fusion = nn.Conv3d(out_channels, out_channels, 3, padding=1)

    def forward(self, coronal, sagittal, D):
        B = coronal.shape[0]
        W = coronal.shape[2]
        H = sagittal.shape[2]

        cor_feat = self.coronal_encoder(coronal)
        sag_feat = self.sagittal_encoder(sagittal)

        cor_vol = cor_feat.permute(0, 1, 3, 2).unsqueeze(3).expand(-1, -1, -1, H, -1)
        sag_vol = sag_feat.permute(0, 1, 3, 2).unsqueeze(4).expand(-1, -1, -1, -1, W)

        return F.relu(self.fusion(cor_vol + sag_vol))

class ResidualBlock3D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(channels, channels, 3, padding=1),
            nn.GroupNorm(8, channels),
            nn.ReLU(),
            nn.Conv3d(channels, channels, 3, padding=1),
            nn.GroupNorm(8, channels),
        )

    def forward(self, x):
        return F.relu(x + self.block(x))


class UNet3D_Diffusion(nn.Module):
    def __init__(self, cond_channels=16):
        super().__init__()
        self.time_mlp = nn.Sequential(
            TimeEmbedding(64),
            nn.Linear(64, 64),
            nn.ReLU()
        )
        self.time_proj1    = nn.Linear(64, 32)
        self.time_proj2    = nn.Linear(64, 64)
        self.time_proj_mid = nn.Linear(64, 64)

        self.enc1 = nn.Conv3d(3 + cond_channels, 32, 3, padding=1)
        self.enc2 = nn.Conv3d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool3d(2)

        self.mid     = nn.Conv3d(64, 64, 3, padding=1)
        self.mid_res = ResidualBlock3D(64)

        self.dec1 = nn.ConvTranspose3d(64, 32, 2, stride=2)
        self.out  = nn.Conv3d(32, 3, 3, padding=1)

    def forward(self, x, cond, t):
        # x:    (B, 3, H, W, D) → (B, 3, D, H, W)
        # cond: (B, C, D, H, W) — already correct from SliceToVolume
        x     = x.permute(0, 1, 4, 2, 3).contiguous()    # ← ADD
        cond  = cond.permute(0, 1, 2, 3, 4).contiguous()  # no-op, already (B,C,D,H,W)
        t = t.float() / 1000.0
        t_emb = self.time_mlp(t)
        x     = torch.cat([x, cond], dim=1)

        scale1 = self.time_proj1(t_emb)[:, :, None, None, None]
        x1 = F.relu(self.enc1(x) * (1 + scale1))
        x2 = F.relu(self.enc2(self.pool(x1)) + self.time_proj2(t_emb)[:, :, None, None, None])

        x_mid = F.relu(self.mid(x2) + self.time_proj_mid(t_emb)[:, :, None, None, None])
        x_mid = self.mid_res(x_mid)

        x = self.dec1(x_mid) + x1
        x = self.out(x)
        return x.permute(0, 1, 3, 4, 2).contiguous()      # ← ADD: back to (B, 3, H, W, D)


class DiffusionModelManager(nn.Module):
    def __init__(self, cond_channels=16):
        super().__init__()
        self.slice_to_vol = SliceToVolume(out_channels=cond_channels)
        self.unet         = UNet3D_Diffusion(cond_channels=cond_channels)

    def forward(self, x_t, coronal_2d, sagittal_2d, t):
        D       = x_t.shape[-1]
        cond_3d = self.slice_to_vol(coronal_2d, sagittal_2d, D)
        return self.unet(x_t, cond_3d, t)


In [7]:
# ── Loss ──────────────────────────────────────────────────────────────────────
def compute_gradient_loss(field, penalty='l2'):
    dh = torch.abs(field[:, :, 1:, :,  :] - field[:, :, :-1, :,  :])
    dw = torch.abs(field[:, :, :,  1:, :] - field[:, :, :,  :-1, :])
    dd = torch.abs(field[:, :, :,  :, 1:] - field[:, :, :,  :, :-1])
    if penalty == 'l2':
        dh, dw, dd = dh**2, dw**2, dd**2
    return (dh.mean() + dw.mean() + dd.mean()) / 3


def diffusion_loss(model, diffusion, x0, cond_coronal, cond_sagittal,
                   lambda_smooth=1e-4):
    B     = x0.shape[0]
    t     = torch.randint(0, diffusion.timesteps, (B,), device=x0.device)
    noise = torch.randn_like(x0)
    x_t   = diffusion.q_sample(x0, t, noise)

    noise_pred = model(x_t, cond_coronal, cond_sagittal, t)
    mse_loss   = F.mse_loss(noise_pred, noise)

    s_ab = diffusion.sqrt_alphas_bar[t].view(B, 1, 1, 1, 1)
    s_om = diffusion.sqrt_one_minus_alphas_bar[t].view(B, 1, 1, 1, 1)
    pred_x0    = (x_t - s_om * noise_pred) / s_ab
    grad_loss  = compute_gradient_loss(pred_x0)
    total_loss = mse_loss + lambda_smooth * grad_loss

    return total_loss, mse_loss, grad_loss


def get_smooth_lambda(epoch, total_epochs, initial_lambda=1e-4, final_lambda=1e-6):
    if epoch < 200:
        return initial_lambda
    decay_range = total_epochs - 200
    step = (initial_lambda - final_lambda) / decay_range
    return max(initial_lambda - step * (epoch - 200), final_lambda)



In [30]:
# ── Training ──────────────────────────────────────────────────────────────────
device    = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
diffusion = DiffusionSchedule(timesteps=1000, device=device)
model     = DiffusionModelManager(cond_channels=16).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=450, eta_min=1e-6)

train_loss_history,   val_loss_history   = [], []
train_mse_history,    val_mse_history    = [], []
train_smooth_history, val_smooth_history = [], []

num_epochs    = 450
best_val_loss = float("inf")
start_epoch   = 1

resume_path = os.path.join(SAVE_DIR, "latest_checkpoint.pth")
if os.path.exists(resume_path):
    checkpoint = torch.load(resume_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint.get('scheduler_state_dict',
                                              scheduler.state_dict()))
    best_val_loss        = checkpoint['best_val_loss']
    train_loss_history   = checkpoint.get('train_loss_history',   [])
    val_loss_history     = checkpoint.get('val_loss_history',     [])
    train_mse_history    = checkpoint.get('train_mse_history',    [])
    val_mse_history      = checkpoint.get('val_mse_history',      [])
    train_smooth_history = checkpoint.get('train_smooth_history', [])
    val_smooth_history   = checkpoint.get('val_smooth_history',   [])
    start_epoch          = checkpoint['epoch'] + 1
    print(f"Resumed from epoch {checkpoint['epoch']} | Best val loss: {best_val_loss:.4f}")
else:
    print("No checkpoint found — starting from scratch")

for epoch in range(start_epoch, num_epochs + 1):
    current_lambda = get_smooth_lambda(epoch, num_epochs)

    model.train()
    train_total = train_mse = train_smooth = 0.0
    for cond_coronal, cond_sagittal, dvf, slice_pos in train_loader:
        dvf           = dvf.to(device)
        cond_coronal  = cond_coronal.to(device)
        cond_sagittal = cond_sagittal.to(device)

        total_loss, mse_loss, smooth_loss = diffusion_loss(
            model, diffusion, dvf, cond_coronal, cond_sagittal,
            lambda_smooth=current_lambda
        )
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_total  += total_loss.item()
        train_mse    += mse_loss.item()
        train_smooth += smooth_loss.item() * current_lambda

    n_train       = len(train_loader)
    train_total  /= n_train
    train_mse    /= n_train
    train_smooth /= n_train
    train_loss_history.append(train_total)
    train_mse_history.append(train_mse)
    train_smooth_history.append(train_smooth)

    model.eval()
    val_total = val_mse = val_smooth = 0.0
    with torch.no_grad():
        for cond_coronal, cond_sagittal, dvf, slice_pos in val_loader:
            dvf           = dvf.to(device)
            cond_coronal  = cond_coronal.to(device)
            cond_sagittal = cond_sagittal.to(device)

            total_loss, mse_loss, smooth_loss = diffusion_loss(
                model, diffusion, dvf, cond_coronal, cond_sagittal,
                lambda_smooth=current_lambda
            )
            val_total  += total_loss.item()
            val_mse    += mse_loss.item()
            val_smooth += smooth_loss.item() * current_lambda

    n_val       = len(val_loader)
    val_total  /= n_val
    val_mse    /= n_val
    val_smooth /= n_val
    val_loss_history.append(val_total)
    val_mse_history.append(val_mse)
    val_smooth_history.append(val_smooth)

    scheduler.step()

    is_best = val_total < best_val_loss
    if is_best:
        best_val_loss = val_total
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pth"))

    torch.save({
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_loss':        best_val_loss,
        'train_loss_history':   train_loss_history,
        'val_loss_history':     val_loss_history,
        'train_mse_history':    train_mse_history,
        'val_mse_history':      val_mse_history,
        'train_smooth_history': train_smooth_history,
        'val_smooth_history':   val_smooth_history,
        'current_lambda':       current_lambda,
    }, os.path.join(SAVE_DIR, "latest_checkpoint.pth"))

    # ── Visual sample every 25 epochs ────────────────────────────────────
    if epoch % 25 == 0:
        print(f"  Running fast repaint visualization...")
        log_dvf_sample(model, diffusion, val_loader, device, epoch)
        print(f"  Done.")

    tag = " *** BEST ***" if is_best else ""
    print(
        f"Epoch {epoch:03d}/{num_epochs} | λ={current_lambda:.2e} | "
        f"LR={scheduler.get_last_lr()[0]:.2e} | "
        f"Train [Total={train_total:.5f} MSE={train_mse:.5f} Smooth={train_smooth:.5f}] | "
        f"Val   [Total={val_total:.5f} MSE={val_mse:.5f} Smooth={val_smooth:.5f}]"
        f"{tag}"
    )

    if epoch % 10 == 0:
        epochs_ax = range(1, len(train_loss_history) + 1)
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].plot(epochs_ax, train_loss_history, label="Train")
        axes[0].plot(epochs_ax, val_loss_history,   label="Val")
        axes[0].set_title("Total Loss"); axes[0].legend(); axes[0].grid(True)
        axes[1].plot(epochs_ax, train_mse_history,  label="Train")
        axes[1].plot(epochs_ax, val_mse_history,    label="Val")
        axes[1].set_title("MSE Loss");   axes[1].legend(); axes[1].grid(True)
        axes[2].plot(epochs_ax, train_smooth_history, label="Train")
        axes[2].plot(epochs_ax, val_smooth_history,   label="Val")
        axes[2].set_title("Smoothness Loss (λ-weighted)")
        axes[2].legend(); axes[2].grid(True)
        plt.suptitle(f"Training curves — epoch {epoch}", fontsize=13)
        plt.tight_layout()
        plt.savefig(os.path.join(VIS_DIR, "loss_curves.png"),
                    dpi=120, bbox_inches='tight')
        plt.close()

print("Training complete.")

Using device: cuda
No checkpoint found — starting from scratch



KeyboardInterrupt



In [ ]:
@torch.no_grad()
def sample_dvf_repaint(model, diffusion, cond_coronal, cond_sagittal,
                       known_coronal, known_sagittal, shape,
                       resampling_steps=5):
    B, C, H, W, D = shape
    mid_h = H // 2
    mid_w = W // 2

    mask = torch.zeros(shape, device=cond_coronal.device)
    mask[:, :, mid_h, :, :] = 1.0
    mask[:, :, :, mid_w, :] = 1.0

    x_known_0 = torch.zeros(shape, device=cond_coronal.device)
    x_known_0[:, :, mid_h, :, :] = known_coronal
    x_known_0[:, :, :, mid_w, :] = known_sagittal

    x = torch.randn(shape, device=cond_coronal.device)

    for t in tqdm(reversed(range(diffusion.timesteps)),
                  total=diffusion.timesteps, desc="Sampling", leave=False):
        for u in range(resampling_steps):
            tt      = torch.full((B,), t, device=cond_coronal.device, dtype=torch.long)
            eps_hat = model(x, cond_coronal, cond_sagittal, tt)

            beta_t    = diffusion.betas[t].view(1,1,1,1,1)
            alpha_t   = diffusion.alphas[t].view(1,1,1,1,1)
            abar_t    = diffusion.alpha_bar[t].view(1,1,1,1,1)
            abar_prev = (diffusion.alpha_bar[t-1].view(1,1,1,1,1)
                         if t > 0 else torch.ones_like(abar_t))

            x0_hat = (x - torch.sqrt(1 - abar_t) * eps_hat) / torch.sqrt(abar_t)
            coef1  = torch.sqrt(abar_prev) * beta_t / (1 - abar_t)
            coef2  = torch.sqrt(alpha_t) * (1 - abar_prev) / (1 - abar_t)
            mean   = coef1 * x0_hat + coef2 * x

            if t > 0:
                var            = beta_t * (1 - abar_prev) / (1 - abar_t)
                x_gen          = mean + torch.sqrt(var) * torch.randn_like(x)
                noise_known    = torch.randn_like(x_known_0)
                x_known_t_prev = (torch.sqrt(abar_prev) * x_known_0
                                  + torch.sqrt(1 - abar_prev) * noise_known)
                x = mask * x_known_t_prev + (1 - mask) * x_gen
                if u < resampling_steps - 1:
                    x = torch.sqrt(1 - beta_t) * x + torch.sqrt(beta_t) * torch.randn_like(x)
            else:
                x = mask * x_known_0 + (1 - mask) * mean

    return x

In [8]:
# ── Overfit Sanity Check ───────────────────────────────────────────────────────
# Trains on 5 fixed samples and evaluates on the same data.
# Goal: loss should drop steadily → confirms the model/pipeline is working.
# If loss does NOT decrease, something is broken (gradients, shapes, etc.)

print("=" * 60)
print("OVERFIT SANITY CHECK")
print("5 samples | 2000 epochs | train == eval data | 128x128x64")
print("=" * 60)

N_OVERFIT      = 5
OVERFIT_EPOCHS = 2000
OVERFIT_LR     = 1e-4

# ── Grab 5 fixed samples from the train dataset ───────────────────────────────
overfit_indices = list(range(N_OVERFIT))
overfit_subset  = torch.utils.data.Subset(train_dataset, overfit_indices)
overfit_loader  = DataLoader(overfit_subset, batch_size=N_OVERFIT,
                             shuffle=False, num_workers=0)

# ── Fresh model & optimizer (don't touch the real ones) ───────────────────────
device_of    = "cuda" if torch.cuda.is_available() else "cpu"
diffusion_of = DiffusionSchedule(timesteps=1000, device=device_of)
model_of     = DiffusionModelManager(cond_channels=16).to(device_of)
optimizer_of = torch.optim.Adam(model_of.parameters(), lr=OVERFIT_LR)

# ── Pre-load and downsample the 5 samples ────────────────────────────────────
# Full resolution: DVF (3, 256, 256, 128) | coronal (3, 256, 128) | sagittal (3, 256, 128)
# Target:         DVF (3, 128, 128,  64) | coronal (3, 128,  64) | sagittal (3, 128,  64)

batch_of     = next(iter(overfit_loader))
cond_cor_of  = batch_of[0].to(device_of)   # (B, 3, 256, 128)
cond_sag_of  = batch_of[1].to(device_of)   # (B, 3, 256, 128)
dvf_of       = batch_of[2].to(device_of)   # (B, 3, 256, 256, 128)


# Downsample DVF: (B, 3, H, W, D) → (B, 3, 128, 128, 64)
dvf_of = F.interpolate(dvf_of, size=(128, 128, 64), mode="trilinear", align_corners=False)

# Downsample coronal conditioning: (B, 3, W, D) → (B, 3, 128, 64)
cond_cor_of = F.interpolate(cond_cor_of, size=(128, 64), mode="bilinear", align_corners=False)

# Downsample sagittal conditioning: (B, 3, H, D) → (B, 3, 128, 64)
cond_sag_of = F.interpolate(cond_sag_of, size=(128, 64), mode="bilinear", align_corners=False)

dvf_of = (dvf_of - dvf_of.mean()) / dvf_of.std()

print(f"DVF shape:      {tuple(dvf_of.shape)}")
print(f"Coronal shape:  {tuple(cond_cor_of.shape)}")
print(f"Sagittal shape: {tuple(cond_sag_of.shape)}")

of_loss_history = []

for epoch in range(1, OVERFIT_EPOCHS + 1):
    model_of.train()
    total_loss, mse_loss, smooth_loss = diffusion_loss(
        model_of, diffusion_of, dvf_of, cond_cor_of, cond_sag_of,
        lambda_smooth=1e-4
    )
    optimizer_of.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model_of.parameters(), max_norm=1.0)
    optimizer_of.step()

    of_loss_history.append(total_loss.item())

    if epoch % 100 == 0 or epoch == 1:
        print(f"  [Overfit] Epoch {epoch:04d}/{OVERFIT_EPOCHS} | "
              f"Total={total_loss.item():.5f}  MSE={mse_loss.item():.5f}  "
              f"Smooth={smooth_loss.item():.5f}")

# ── Plot the overfit curve ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(of_loss_history, label="Overfit loss (train==eval)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Total loss")
ax.set_title("Overfit sanity check — should decrease steadily")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, "overfit_check.png"), dpi=120, bbox_inches="tight")
plt.show()
print(f"\nOverfit check saved to {VIS_DIR}/overfit_check.png")

# ── Pass/fail heuristic ───────────────────────────────────────────────────────
loss_start = of_loss_history[0]
loss_end   = of_loss_history[-1]
reduction  = (loss_start - loss_end) / loss_start * 100
print(f"\nLoss reduction: {loss_start:.5f} → {loss_end:.5f}  ({reduction:.1f}%)")
if reduction > 30:
    print("✓ PASSED — model is learning, pipeline looks healthy.")
else:
    print("✗ WARNING — loss barely moved. Check gradients, LR, data shapes.")

print("=" * 60)
print("Overfit check done. Proceeding to real training...")
print("=" * 60)

OVERFIT SANITY CHECK
5 samples | 2000 epochs | train == eval data | 128x128x64
DVF shape:      (5, 3, 128, 128, 64)
Coronal shape:  (5, 3, 128, 64)
Sagittal shape: (5, 3, 128, 64)
  [Overfit] Epoch 0001/2000 | Total=1.15877  MSE=1.11261  Smooth=461.53769
  [Overfit] Epoch 0100/2000 | Total=0.95037  MSE=0.75612  Smooth=1942.47351
  [Overfit] Epoch 0200/2000 | Total=0.30609  MSE=0.28336  Smooth=227.33623
  [Overfit] Epoch 0300/2000 | Total=0.22517  MSE=0.21627  Smooth=88.92602
  [Overfit] Epoch 0400/2000 | Total=0.08381  MSE=0.07298  Smooth=108.30286
  [Overfit] Epoch 0500/2000 | Total=0.21172  MSE=0.21171  Smooth=0.08933
  [Overfit] Epoch 0600/2000 | Total=0.02662  MSE=0.02455  Smooth=20.71459
  [Overfit] Epoch 0700/2000 | Total=0.02822  MSE=0.02188  Smooth=63.36734
  [Overfit] Epoch 0800/2000 | Total=0.21734  MSE=0.21729  Smooth=0.45353
  [Overfit] Epoch 0900/2000 | Total=0.17205  MSE=0.16789  Smooth=41.55680
  [Overfit] Epoch 1000/2000 | Total=0.16016  MSE=0.16013  Smooth=0.31367
  [O

In [9]:
# ── Overfit Sanity Check — Reconstruct DVF via reverse diffusion ───────────────
# Starts from pure noise and denoises all the way to t=0 using the overfit
# model. Since the model was conditioned on these exact samples, the output
# should closely match the GT DVFs.

print("Running full reverse diffusion (no repaint masking)...")

model_of.eval()
B, C, H, W, D = dvf_of.shape
mid_d = D // 2

@torch.no_grad()
def sample_simple(model, diffusion, cond_coronal, cond_sagittal, shape):
    """Plain DDPM reverse diffusion, no masking."""
    x = torch.randn(shape, device=cond_coronal.device)
    for t in tqdm(reversed(range(diffusion.timesteps)),
                  total=diffusion.timesteps, desc="Denoising", leave=False):
        tt      = torch.full((shape[0],), t, device=x.device, dtype=torch.long)
        eps_hat = model(x, cond_coronal, cond_sagittal, tt)

        beta_t    = diffusion.betas[t].view(1,1,1,1,1)
        alpha_t   = diffusion.alphas[t].view(1,1,1,1,1)
        abar_t    = diffusion.alpha_bar[t].view(1,1,1,1,1)
        abar_prev = (diffusion.alpha_bar[t-1].view(1,1,1,1,1)
                     if t > 0 else torch.ones_like(abar_t))

        x0_hat = (x - torch.sqrt(1 - abar_t) * eps_hat) / torch.sqrt(abar_t)
        coef1  = torch.sqrt(abar_prev) * beta_t / (1 - abar_t)
        coef2  = torch.sqrt(alpha_t) * (1 - abar_prev) / (1 - abar_t)
        mean   = coef1 * x0_hat + coef2 * x

        if t > 0:
            var = beta_t * (1 - abar_prev) / (1 - abar_t)
            x   = mean + torch.sqrt(var) * torch.randn_like(x) #con il commento diventa DDPM, senza DDIM
        else:
            x = mean
    return x

dvf_sampled = sample_simple(
    model_of, diffusion_of, cond_cor_of, cond_sag_of,
    shape=(B, C, H, W, D)
)

# ── Plot GT vs reconstructed vs error ─────────────────────────────────────────
dvf_sampled_np = dvf_sampled.cpu().numpy()
dvf_gt_np      = dvf_of.cpu().numpy()

fig, axes = plt.subplots(N_OVERFIT, 3, figsize=(13, N_OVERFIT * 3.5))

for i in range(N_OVERFIT):
    gt_sl   = dvf_gt_np[i, 0, :, :, mid_d]
    pred_sl = dvf_sampled_np[i, 0, :, :, mid_d]
    err_sl  = np.abs(gt_sl - pred_sl)

    vmin, vmax = gt_sl.min(), gt_sl.max()

    axes[i, 0].imshow(gt_sl,   cmap="RdBu_r", vmin=vmin, vmax=vmax)
    axes[i, 0].set_title(f"Sample {i} — DVF x (GT)")

    axes[i, 1].imshow(pred_sl, cmap="RdBu_r", vmin=vmin, vmax=vmax)
    axes[i, 1].set_title(f"Sample {i} — DVF x (reconstructed)")

    im = axes[i, 2].imshow(err_sl, cmap="hot")
    axes[i, 2].set_title(f"Sample {i} — |Error|  (mean={err_sl.mean():.3f})")
    plt.colorbar(im, ax=axes[i, 2], fraction=0.046)

    for ax in axes[i]:
        ax.axis("off")

plt.suptitle("Overfit check — full reverse diffusion, GT vs reconstructed\n"
             "(mid axial slice, channel 0)", fontsize=13)
plt.tight_layout()
fname = os.path.join(VIS_DIR, "overfit_viz_reconstructed.png")
plt.savefig(fname, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved to {fname}")

Running full reverse diffusion (no repaint masking)...


Denoising:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved to /mimer/NOBACKUP/groups/caim1/dafne/visual/overfit_viz_reconstructed.png


In [11]:
# ── Overfit Sanity Check — Reconstruct DVF via reverse diffusion ───────────────
# Starts from pure noise and denoises all the way to t=0 using the overfit
# model. Since the model was conditioned on these exact samples, the output
# should closely match the GT DVFs.

print("Running full reverse diffusion (no repaint masking)...")

model_of.eval()
B, C, H, W, D = dvf_of.shape
mid_d = D // 2

@torch.no_grad()
def sample_simple(model, diffusion, cond_coronal, cond_sagittal, shape):
    """Plain DDPM reverse diffusion, no masking."""
    x = torch.randn(shape, device=cond_coronal.device)
    for t in tqdm(reversed(range(diffusion.timesteps)),
                  total=diffusion.timesteps, desc="Denoising", leave=False):
        tt      = torch.full((shape[0],), t, device=x.device, dtype=torch.long)
        eps_hat = model(x, cond_coronal, cond_sagittal, tt)

        beta_t    = diffusion.betas[t].view(1,1,1,1,1)
        alpha_t   = diffusion.alphas[t].view(1,1,1,1,1)
        abar_t    = diffusion.alpha_bar[t].view(1,1,1,1,1)
        abar_prev = (diffusion.alpha_bar[t-1].view(1,1,1,1,1)
                     if t > 0 else torch.ones_like(abar_t))

        x0_hat = (x - torch.sqrt(1 - abar_t) * eps_hat) / torch.sqrt(abar_t)
        coef1  = torch.sqrt(abar_prev) * beta_t / (1 - abar_t)
        coef2  = torch.sqrt(alpha_t) * (1 - abar_prev) / (1 - abar_t)
        mean   = coef1 * x0_hat + coef2 * x

        if t > 0:
            var = beta_t * (1 - abar_prev) / (1 - abar_t)
            x   = mean + torch.sqrt(var) * torch.randn_like(x)
        else:
            x = mean
    return x

dvf_sampled = sample_simple(
    model_of, diffusion_of, cond_cor_of, cond_sag_of,
    shape=(B, C, H, W, D)
)

# ── Plot GT vs reconstructed vs error ─────────────────────────────────────────
dvf_sampled_np = dvf_sampled.cpu().numpy()
dvf_gt_np      = dvf_of.cpu().numpy()

fig, axes = plt.subplots(N_OVERFIT, 3, figsize=(13, N_OVERFIT * 3.5))

for i in range(N_OVERFIT):
    gt_sl   = dvf_gt_np[i, 0, :, :, mid_d]
    pred_sl = dvf_sampled_np[i, 0, :, :, mid_d]
    err_sl  = np.abs(gt_sl - pred_sl)

    vmin, vmax = gt_sl.min(), gt_sl.max()

    axes[i, 0].imshow(gt_sl,   cmap="RdBu_r", vmin=vmin, vmax=vmax)
    axes[i, 0].set_title(f"Sample {i} — DVF x (GT)")

    axes[i, 1].imshow(pred_sl, cmap="RdBu_r", vmin=vmin, vmax=vmax)
    axes[i, 1].set_title(f"Sample {i} — DVF x (reconstructed)")

    im = axes[i, 2].imshow(err_sl, cmap="hot")
    axes[i, 2].set_title(f"Sample {i} — |Error|  (mean={err_sl.mean():.3f})")
    plt.colorbar(im, ax=axes[i, 2], fraction=0.046)

    for ax in axes[i]:
        ax.axis("off")

plt.suptitle("Overfit check — full reverse diffusion, GT vs reconstructed\n"
             "(mid axial slice, channel 0)", fontsize=13)
plt.tight_layout()
fname = os.path.join(VIS_DIR, "overfit_viz_reconstructed.png")
plt.savefig(fname, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved to {fname}")

Running full reverse diffusion (no repaint masking)...


Denoising:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved to /mimer/NOBACKUP/groups/caim1/dafne/visual/overfit_viz_reconstructed.png


In [14]:
print("noise std:", noise_sl.std())
print("pred std :", pred_sl.std())

noise std: 0.99357545
pred std : 20.185337


In [15]:
corr = np.corrcoef(noise_sl.flatten(), pred_sl.flatten())[0,1]
print("corr:", corr)

corr: -0.002847441519429319
